# Cvičení — cross-validation splits

Cíl: napsat funkci, která každému vzorku přiřadí **číslo foldu** (0 až K−1) tak, aby splňovala dvě pravidla:

1. **Target vzorky** — žádné dvě vzorky ze stejné `session_id` nesmí skončit v různých foldech.
2. **Non-target vzorky** — žádné dvě vzorky od stejného speakera (`identity`) nesmí skončit v různých foldech.

Pokud tohle zvládneš, tak:
- audio model se nebude umět "cheatnout" tím, že si v trainu zapamatuje pozadí session,
- image model se nenaučí rozpoznávat osvětlení místo obličeje,
- tvoje CV skóre bude **poctivý odhad** toho, jak to dopadne na ostrých datech.

V notebooku jsou bloky `# TODO`, které budeš dopisovat. Pod každým je **validation cell**, co tvoji implementaci zkontroluje a buď ti řekne `✓ OK` nebo vypíše, co neklape. Nepodvádět — validace ti řekne **co**, ale ne **jak**. ;)

Jakmile to projde, převedeme tvoji finální funkci do `src/data/splits.py`.

## 0. Setup — manifest (hotové, nic neměň)

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

DATA = Path("../data").resolve()
SPLITS = {
    "target_train":     {"label": 1},
    "target_dev":       {"label": 1},
    "non_target_train": {"label": 0},
    "non_target_dev":   {"label": 0},
}
NAME_RE = re.compile(r"^(?P<identity>[fm]\d+)_(?P<session>\d+)_(?P<prompt>[a-z]\d+)_i(?P<inst>\d+)_(?P<take>\d+)$")

rows = []
for folder, meta in SPLITS.items():
    for p in sorted((DATA / folder).glob("*.png")):  # one row per sample (not per file)
        m = NAME_RE.match(p.stem)
        assert m, p.name
        rows.append({**m.groupdict(), "label": meta["label"], "stem": p.stem})

manifest = pd.DataFrame(rows)
manifest["session_id"] = manifest["identity"] + "_" + manifest["session"]
print(f"{len(manifest)} samples, {manifest.label.sum()} target, {(1 - manifest.label).sum()} non-target")
manifest.head()

## 1. Cvičení 1 — definuj skupinu pro každý vzorek

Potřebujeme jeden sloupec `group`, který říká *"tenhle vzorek patří do této group"*. Pravidlo:

- Když je `label == 1` (target) → `group = session_id` (jako `m431_01`).
- Když je `label == 0` (non-target) → `group = identity` (jako `f401`).

**Proč dva různé zdroje group?** Target má málo lidí (jen 1) a hodně sessions. Non-target má hodně lidí a málo sessions. Chceme simulovat:
- u targetu: *"uvidíme toho člověka v úplně nové situaci"*,
- u non-targetu: *"uvidíme úplně cizího člověka"*.

Výsledek uložíme do sloupce `manifest['group']`.

In [ ]:
# TODO: vytvoř sloupec manifest['group'] podle pravidla výše.
# Tip: np.where(cond, a, b) vrátí pole, kde vybírá z a nebo b podle podmínky.

manifest["group"] = None  # <-- nahraď

manifest[["stem", "label", "identity", "session_id", "group"]].head(10)

In [ ]:
# --- VALIDATION 1 ---
def check_group_column(df):
    problems = []
    if df["group"].isna().any():
        problems.append("sloupec group obsahuje None/NaN")
    t = df[df.label == 1]
    if not (t["group"] == t["session_id"]).all():
        problems.append("target rows: group se nerovná session_id")
    nt = df[df.label == 0]
    if not (nt["group"] == nt["identity"]).all():
        problems.append("non-target rows: group se nerovná identity")
    return problems

p = check_group_column(manifest)
print("✓ OK" if not p else "✗ FAIL:\n  - " + "\n  - ".join(p))

## 2. Cvičení 2 — rozděl skupiny do foldů

Teď máš sloupec `group`. Úkol: **každé unikátní skupině přiřadit číslo foldu** (`0..K-1`) tak, aby:

- target skupiny (session_id typu `m431_01`) byly rozházené napříč foldy,
- non-target skupiny (identity typu `f401`) taky,
- label balance se neroztrhal — v každém foldu by měl být aspoň 1 target a nějaké non-targety.

Nejjednodušší technika: pro každý label zvlášť **zamíchej skupiny** a přiřaď jim foldy round-robin (`i % K`).

Napiš funkci `assign_folds(manifest, n_splits, seed) -> pd.Series` (index = stejný jako manifest, hodnota = číslo foldu).

**Hint:**
```
rng = np.random.default_rng(seed)
fold_of_group = {}  # dict: group_name -> fold_id
for label, sub in manifest.groupby("label"):
    unique_groups = sub["group"].unique()
    rng.shuffle(unique_groups)
    for i, g in enumerate(unique_groups):
        fold_of_group[g] = i % n_splits
# pak už jen namapovat na celý dataframe
```

In [ ]:
def assign_folds(df: pd.DataFrame, n_splits: int = 5, seed: int = 0) -> pd.Series:
    """
    Vrať pd.Series, kde index odpovídá df.index a hodnota je fold_id (0..n_splits-1).
    Vzorky se stejnou `group` MUSÍ mít stejný fold.
    Pro každý label rozděluj skupiny separátně (jinak ti jeden label přebije druhý).
    """
    # TODO: implementace
    return pd.Series(0, index=df.index)  # <-- nahraď

manifest["fold"] = assign_folds(manifest, n_splits=5, seed=0)
manifest["fold"].value_counts().sort_index()

In [ ]:
# --- VALIDATION 2 ---
def check_folds(df, n_splits=5):
    problems = []

    # (a) každá group má jen jeden fold
    bad = df.groupby("group")["fold"].nunique()
    leaky = bad[bad > 1]
    if len(leaky):
        problems.append(f"{len(leaky)} skupin je rozdělených mezi foldy: {list(leaky.index[:5])}")

    # (b) všechny foldy existují
    folds_used = sorted(df["fold"].unique())
    if folds_used != list(range(n_splits)):
        problems.append(f"očekávám foldy {list(range(n_splits))}, dostal jsem {folds_used}")

    # (c) každý fold má aspoň 1 target a aspoň nějaké non-targety
    per_fold = df.groupby(["fold", "label"]).size().unstack(fill_value=0)
    missing_pos = per_fold.index[per_fold.get(1, 0) == 0].tolist()
    missing_neg = per_fold.index[per_fold.get(0, 0) == 0].tolist()
    if missing_pos:
        problems.append(f"foldy bez target vzorků: {missing_pos}")
    if missing_neg:
        problems.append(f"foldy bez non-target vzorků: {missing_neg}")

    return problems, per_fold

p, table = check_folds(manifest, n_splits=5)
print(table)
print()
print("✓ OK" if not p else "✗ FAIL:\n  - " + "\n  - ".join(p))

**Co se stane, když dáš `n_splits=5` a máš jen 3 target sessions?**

Zkus! Validace tě upozorní. Je to **feature, ne bug** — říká ti, že pro target data má smysl max tolik foldů, kolik je unikátních sessions (protože se nedá sedět na víc židlích).

Proto dole zkoušíme různé `n_splits` a uvidíme, kolik foldů má vůbec smysl.

In [ ]:
for k in [3, 4, 5, 6]:
    f = assign_folds(manifest, n_splits=k, seed=0)
    manifest["fold"] = f
    p, table = check_folds(manifest, n_splits=k)
    status = "OK" if not p else "FAIL"
    print(f"n_splits={k}: {status}")
    if p:
        for x in p:
            print(f"   - {x}")

## 3. Cvičení 3 — generátor train/val indexů

Nakonec chceme použitelné API pro modely. Napiš generátor:

```python
for fold_id, train_idx, val_idx in iter_folds(manifest, n_splits=3):
    X_train, y_train = features[train_idx], y[train_idx]
    X_val,   y_val   = features[val_idx],   y[val_idx]
    ...
```

- `train_idx` = numpy array indexů vzorků, co **nejsou** v aktuálním foldu
- `val_idx` = numpy array indexů vzorků, co **jsou** v aktuálním foldu

Tohle je tvar, který sklearn používá v `KFold.split()`, a je to forma, kterou **přímo zkopírujeme do `src/data/splits.py`**.

In [ ]:
def iter_folds(df: pd.DataFrame, n_splits: int = 3, seed: int = 0):
    """
    Yielduj tuples (fold_id, train_idx, val_idx) pro každý fold.
    Reuse: assign_folds().
    """
    # TODO
    yield from ()  # <-- nahraď

# rychlá ukázka
for fold_id, train_idx, val_idx in iter_folds(manifest, n_splits=3):
    print(f"fold {fold_id}: train={len(train_idx):3d}, val={len(val_idx):3d}")

In [ ]:
# --- VALIDATION 3 ---
def check_iter(df, n_splits=3):
    problems = []
    seen_val = np.zeros(len(df), dtype=int)
    fold_ids = []
    for fold_id, train_idx, val_idx in iter_folds(df, n_splits=n_splits):
        fold_ids.append(fold_id)
        if set(train_idx) & set(val_idx):
            problems.append(f"fold {fold_id}: train a val se překrývají")
        if set(train_idx) | set(val_idx) != set(df.index):
            problems.append(f"fold {fold_id}: train ∪ val není celé df")
        train_groups = set(df.loc[train_idx, "group"])
        val_groups   = set(df.loc[val_idx,   "group"])
        if train_groups & val_groups:
            problems.append(f"fold {fold_id}: group leak mezi train a val")
        seen_val[val_idx] += 1
    if fold_ids != list(range(n_splits)):
        problems.append(f"fold_ids: {fold_ids}, očekávám {list(range(n_splits))}")
    if (seen_val != 1).any():
        problems.append(f"{(seen_val != 1).sum()} vzorků nebylo ve val právě jednou")
    return problems

p = check_iter(manifest, n_splits=3)
print("✓ OK" if not p else "✗ FAIL:\n  - " + "\n  - ".join(p))

## 4. Bonus — Leave-One-Session-Out na targetu

Kolik máme cílových sessions? Sečti unikátní `session_id` kde `label == 1`. Jsou to jen 3. Místo 5-fold dává ve skutečnosti nejlepší smysl **3-fold LOSO** — každý fold validuje přesně jednu session targetu, a non-target speakeři se přirozeně rozháže napříč.

**Bonusové cvičení (nepovinné):** napiš variantu `iter_folds_loso(df)`, která yielduje přesně tolik foldů, kolik je target sessions, a v každém foldu je právě jedna target session na valu.

In [ ]:
target_sessions = sorted(manifest.query("label == 1")["session_id"].unique())
print(f"target sessions: {target_sessions} ({len(target_sessions)} foldů dává smysl)")

In [ ]:
def iter_folds_loso(df: pd.DataFrame, seed: int = 0):
    """
    Leave-One-Session-Out na targetu:
    - počet foldů = počet target sessions
    - v i-tém foldu je val = všechny vzorky té i-té target session + alikvótní část non-target speakerů
    - train = zbytek
    
    Pro non-target speakery: zamíchej je a rozděl na K stejně velkých skupin,
    kde K = počet target sessions.
    """
    # TODO (volitelné)
    yield from ()

for fold_id, train_idx, val_idx in iter_folds_loso(manifest):
    val = manifest.loc[val_idx]
    t_sess = sorted(val.query("label == 1")["session_id"].unique())
    nt_ids = sorted(val.query("label == 0")["identity"].unique())
    print(f"fold {fold_id}: val target session = {t_sess}, val non-target speakers = {nt_ids}")

## 5. Hotovo → převedeme do `src/data/splits.py`

Až ti všechny validace vracejí `✓ OK`, dej vědět. Zkopírujeme tvoji finální implementaci do `src/data/splits.py` s čistým API:

```python
from src.data.splits import load_manifest, iter_folds, iter_folds_loso

manifest = load_manifest(Path("data"))
for fold_id, train_idx, val_idx in iter_folds(manifest, n_splits=3):
    ...
```

Tohle API pak budou používat audio, image i fusion systémy — **všechny tři systémy sdílí přesně ten samý fold assignment**, což je load-bearing pro stackingovou fúzi.